In [ ]:
import os
from pathlib import Path

from qiskit import QuantumCircuit
from qiskit.qasm3 import loads as loads_qasm3

from q_rewrite.clients import ModelClient
from q_rewrite.optimizers import Optimizer
from q_rewrite.utilities.logging import get_logger
from q_rewrite.utilities.os import load_env_file
from q_rewrite.verifiers import QiskitVerifier


# load the env vars
load_env_file(project_root_path=Path.cwd())
logger = get_logger()
model_client = ModelClient(
    api_key=os.environ["MODEL_API_KEY"],
    base_url=os.environ["MODEL_API_BASE_URL"],
    logger=logger,
    model=os.environ["MODEL"],
)
optimizer = Optimizer(
    model_client=model_client,
    logger=logger,
    verifier=QiskitVerifier(logger=logger),
)

## 1. Simple Cancellation

In [ ]:
qasm = Path("./examples/01_cancellation.qasm").read_text(encoding="utf-8")
circuit: QuantumCircuit = loads_qasm3(qasm)
circuit.draw()

**Possible optimizations**:
- Remove inverse $X$ gates because $X^{2} = I$.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 2. Non-syntactically Adjacent Gates

In [ ]:
qasm = Path("./examples/02_non_syntactically_adjacent.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Possible optimizations**:
- Remove inverse $X$ gates on $q_0$ because $X^{2} = I$.

> **NOTE:** The $X$ gates on the first qubit are semantically adjacent, are not syntactically adjacent.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 3. Bell Redundancy

In [ ]:
qasm = Path("./examples/03_bell_redundant.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Possible optimizations**:
- Remove inverse $X$ gates on $q_0$ because $X^{2} = I$.
- Remove the $R_{Z}$ rotations on $q_1$ since $R_{Z}(0.25)R_{Z}(-0.25) = I$.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 4. GHZ (3-qubit)

In [ ]:
qasm = Path("./examples/04_ghz3.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Possible optimizations**:
- Remove inverse $Z$ gates on $q_2$ because $Z^{2} = I$.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 5. GHZ (4-qubit)

In [ ]:
qasm = Path("./examples/05_ghz4_redundant.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Optimizations**:
- Remove inverse $X$ gates on $q_1$ because $X^{2} = I$.
- Remove inverse $H$ gates on $q_3$ because $H^{2} = I$.
- Remove inverse $CNOT$ gates on $q_0$ and $q_1$ because $CNOT$ is self-inverse.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 6. QAOA (MaxCut)

In [ ]:
qasm = Path("./examples/06_qaoa_maxcut3.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Optimizations**:
- Each $CNOT$/$R_{Z}$ block is a conjugation by $CNOT$, and since $R_{Z}(\theta)$ is diagonal, so they can be rewritten as controlled $R_{Z}$ operations:
$$
  CNOT(control, target)(I_{control} \otimes R_{Z}(\theta)_{t})CNOT(control, target) = CRZ_(control, target)(\theta)
$$
> **NOTE:** A controlled $R_{Z}$ may not be a cheaper hardware-aware operation than a $CNOT$/$R_{X}$ block.
- Remove the $R_{Z}$ rotations on $q_0$ since $R_{Z}(0.4)R_{Z}(-0.4) = I$.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 7. QFT (3-qubit)

In [ ]:
qasm = Path("./examples/07_qft3_redundant.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Possible optimizations**:
- Merge $RZ$ rotations on $q_1$ since $R_{Z}(0.3)R_{Z}(-0.3) = I$.

> **NOTE:** Since a $SWAP$ gate weighs a significant cost, the final $SWAP$ gate can be removed if the output bit-reversal is handled by classical interpretation or by mapping logical qubits to reversed physical positions. It will be interesting to see if the model catches this!

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 8. QAOA (MaxCut) - back with a vengence!

In [ ]:
qasm = Path("./examples/08_qaoa_maxcut3_p2.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Possible optimizations**:
- Merge the $R_{X}$ rotation blocks on each qubit into one gate since $R_{X}(1.0)R_{X}(0.8) = R_{X}(1.8)$.
> **NOTE:** Since the intervening operations in-between these blocks do not matter because they act on disjoint qubits.
- Like before in [6. QAOA (MaxCut)](#6-qaoa-maxcut), the $CNOT$/$R_{Z}$ blocks can be rewritten as controlled $R_{Z}$ operations.
- Remove inverse $H$ gates on $q_0$ because $H^{2} = I$.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()

## 9. Teleportation

In [ ]:
qasm = Path("./examples/09_teleportation.qasm").read_text(encoding="utf-8")
circuit = loads_qasm3(qasm)
circuit.draw()

**Possible optimizations**:
- In the middle sequence of the first block, there contains a substitution for the $H(q_{1})CNOT(q_{1}, q_{2})H(q_{1})$ into a controlled-$Z$ gate since:
$$
  H(control) CNOT(control, target) H(control) = CZ(control, target)
$$
> **NOTE:** A controlled-$Z$ may not be a cheaper hardware-aware operation than a $H(control) CNOT(control, target) H(control)$ block.

In [ ]:
optimization = optimizer.optimize(circuit=qasm)
circuit = loads_qasm3(optimization.final_circuit)
circuit.draw()